# Antmaze — MCTS-vs-softFloyd ablation (E1a + E1c)

Chạy trên **Kaggle GPU T4** (env MuJoCo cần stack cũ). Đo robustness của planner khi
world-model bị nhiễu: **E1a** (stochastic) + **E1c** (bias cố định + execution feedback).

**Trước khi chạy:** Settings → Accelerator = **GPU T4**, Internet = **ON**;
**Add Data →** output notebook đã train `antmaze_s221_paper` (để restore checkpoint). Eval chỉ cần
`agent.pt`+`algo.pt` (KHÔNG cần replay). AntMaze = long-horizon, planner load-bearing → chỗ MCTS ăn tiền. E1c cần σ MỊN (0.05-0.2): σ nhỏ chưa tạo bẫy, σ lớn (≥0.3) sập hết.

## 1. Code + env (~10–15 phút lần đầu)

In [ ]:
import os
if os.path.isdir('/kaggle/working/latent_landmarks'):
    !cd /kaggle/working/latent_landmarks && git pull -q origin retrain
else:
    !git clone -q -b retrain https://github.com/Jun1801/latent_landmarks.git /kaggle/working/latent_landmarks
if not os.path.isdir('/kaggle/working/wmag'):
    !git clone -q https://github.com/LunjunZhang/world-model-as-a-graph /kaggle/working/wmag
!bash /kaggle/working/latent_landmarks/repro/setup_kaggle.sh

## 2. Restore checkpoint (chỉ agent.pt + algo.pt)
Tìm `<SLUG>` bằng `!ls /kaggle/input/` (là dataset/output bạn vừa Add Data).

In [ ]:
!ls /kaggle/input/
import os, shutil
SLUG = 'PUT-DATASET-SLUG-HERE'          # <-- sửa cho khớp /kaggle/input/
CKPT, ENV = 'antmaze_s221_paper', 'AntMaze-v1'
src = f'/kaggle/input/{SLUG}/experiments/{ENV}/{CKPT}/state'
dst = f'/kaggle/working/experiments/{ENV}/{CKPT}/state'; os.makedirs(dst, exist_ok=True)
for f in ['agent.pt', 'algo.pt']:
    shutil.copy(f'{src}/{f}', f'{dst}/{f}')
print('restored ->', os.listdir(dst))

## 3. Verify GPU + env

In [ ]:
!export PATH=/opt/conda/bin:$PATH; export LD_LIBRARY_PATH=$HOME/.mujoco/mujoco200/bin:/usr/lib/nvidia:${LD_LIBRARY_PATH:-};  conda run -n l3p python -c "import torch,mujoco_py; print('cuda',torch.cuda.is_available())"

## 4. E1a — σ=0 sanity (mcts ≈ soft_floyd ≈ checkpoint success)

In [ ]:
!export PATH=/opt/conda/bin:$PATH; export LD_LIBRARY_PATH=$HOME/.mujoco/mujoco200/bin:/usr/lib/nvidia:${LD_LIBRARY_PATH:-};  conda run -n l3p python /kaggle/working/latent_landmarks/repro/paper_mcts/eval_ablation.py --env antmaze --regime e1a --resume_ckpt antmaze_s221_paper --episodes 2 --n_test_rollouts 20 --sims 100 --sigmas 0

## 5. E1a sweep — phased: classical (fast) → MCTS (slow)

Phase 1 = classical planners (soft_floyd / dijkstra / A\* / greedy), 3 noise seeds, own JSON.
Phase 2 = MCTS variants (slow). AntMaze is long-horizon nav → planner load-bearing (expect MCTS
to separate from soft-Floyd, and Dijkstra/A* to suffer most under noise — single hard path).

In [ ]:
# PHASE 1 — classical planners (fast, single noise seed)
!export PATH=/opt/conda/bin:$PATH; export LD_LIBRARY_PATH=$HOME/.mujoco/mujoco200/bin:/usr/lib/nvidia:${LD_LIBRARY_PATH:-};  conda run -n l3p python /kaggle/working/latent_landmarks/repro/paper_mcts/eval_ablation.py --env antmaze --regime e1a --resume_ckpt antmaze_s221_paper --episodes 3 --n_test_rollouts 30 --sims 100 --latency --planners soft_floyd dijkstra astar greedy --out /kaggle/working/exp_out/antmaze_e1a_classical.json --sigmas 0 5 10 20

In [ ]:
# PHASE 2 — MCTS variants (slow)
!export PATH=/opt/conda/bin:$PATH; export LD_LIBRARY_PATH=$HOME/.mujoco/mujoco200/bin:/usr/lib/nvidia:${LD_LIBRARY_PATH:-};  conda run -n l3p python /kaggle/working/latent_landmarks/repro/paper_mcts/eval_ablation.py --env antmaze --regime e1a --resume_ckpt antmaze_s221_paper --episodes 3 --n_test_rollouts 30 --sims 100 --planners soft_floyd mcts mcts+suffix mcts+pw mcts+bayes --out /kaggle/working/exp_out/antmaze_e1a_mcts.json --sigmas 0 5 10 20

## 6. E1c — σ=0 sanity (soft_floyd/mcts_nofb/mcts_fb trùng nhau)

In [ ]:
!export PATH=/opt/conda/bin:$PATH; export LD_LIBRARY_PATH=$HOME/.mujoco/mujoco200/bin:/usr/lib/nvidia:${LD_LIBRARY_PATH:-};  conda run -n l3p python /kaggle/working/latent_landmarks/repro/paper_mcts/eval_ablation.py --env antmaze --regime e1c --resume_ckpt antmaze_s221_paper --episodes 2 --n_test_rollouts 20 --sims 100 --sigmas 0

## 7. E1c — classical planners (fast), full σ incl. high band

Classical baseline (soft_floyd / dijkstra / A\* / greedy), static, 3 noise seeds, σ up to 0.5.
The MCTS phase for E1c is §7b below.

In [ ]:
# PHASE 1 — classical planners (fast, single noise seed)
!export PATH=/opt/conda/bin:$PATH; export LD_LIBRARY_PATH=$HOME/.mujoco/mujoco200/bin:/usr/lib/nvidia:${LD_LIBRARY_PATH:-};  conda run -n l3p python /kaggle/working/latent_landmarks/repro/paper_mcts/eval_ablation.py --env antmaze --regime e1c --resume_ckpt antmaze_s221_paper --episodes 3 --n_test_rollouts 30 --sims 100 --latency --planners soft_floyd dijkstra astar greedy --out /kaggle/working/exp_out/antmaze_e1c_classical.json --sigmas 0 0.05 0.1 0.15 0.2 0.25 0.3 0.4 0.5

## 7b. E1c — MCTS variants (slow), high-σ band

Phase 2 for E1c: `mcts_nofb` vs `mcts_fb` (execution feedback) vs `soft_floyd`, over the full
σ grid up to **0.5** with `--noise-seeds 0 1 2` ([min,max] band). Finds where the MCTS-over-Floyd
advantage peaks and where all collapse to the floor — the *upper bound*. ~30–60 min.

*(Band varies the noise draw only; still a single **training** seed s221 — firming that needs
seeds 252/173, out of scope.)*

In [ ]:
# PHASE 2 — MCTS variants (slow), E1c high-σ band  (longest cell; ~30-60 min)
!export PATH=/opt/conda/bin:$PATH; export LD_LIBRARY_PATH=$HOME/.mujoco/mujoco200/bin:/usr/lib/nvidia:${LD_LIBRARY_PATH:-};  conda run -n l3p python /kaggle/working/latent_landmarks/repro/paper_mcts/eval_ablation.py --env antmaze --regime e1c --resume_ckpt antmaze_s221_paper --episodes 3 --n_test_rollouts 30 --sims 100 --planners soft_floyd mcts_nofb mcts_fb --noise-seeds 0 1 2 --out /kaggle/working/exp_out/antmaze_e1c_mcts.json --sigmas 0 0.05 0.1 0.15 0.2 0.25 0.3 0.4 0.5

## 8. Dump 3 planner (cho hình plan-comparison + graph wormhole), 1 episode

In [ ]:
!export PATH=/opt/conda/bin:$PATH; export LD_LIBRARY_PATH=$HOME/.mujoco/mujoco200/bin:/usr/lib/nvidia:${LD_LIBRARY_PATH:-};  conda run -n l3p python /kaggle/working/latent_landmarks/repro/paper_mcts/eval_ablation.py --env antmaze --regime e1c --resume_ckpt antmaze_s221_paper --episodes 1 --n_test_rollouts 1 --sims 100 --sigmas 0.2 --dump-plans /kaggle/working/exp_out/antmaze_plans_e1c.json

## 9. Lấy JSON về (để plot local)
`/kaggle/working/exp_out/` được lưu khi **Save Version**. Tải các file → gửi lại tôi plot:
- `antmaze_e1a_classical.json`, `antmaze_e1a_mcts.json` (E1a)
- `antmaze_e1c_classical.json`, `antmaze_e1c_mcts.json` (E1c, σ→0.5, band)
- `antmaze_plans_e1c.json` (hình plan-comparison / wormhole)

Tôi overlay success-vs-σ tất cả planner (band [min,max] cho classical & MCTS multi-seed), định vị
σ\* (gap MCTS−Floyd đỉnh) và σ_floor (sập), rồi cập nhật §7.2/7.3 MCTS.md.

In [ ]:
!ls -la /kaggle/working/exp_out/ 2>/dev/null || echo 'chưa có output'

## Ghi chú
- **σ=0 luôn phải khớp** soft_floyd(clean) — nếu lệch nhiều là port sai, dừng & báo.
- Kết quả in ra bảng success theo σ; copy lại để tổng hợp.
- Chạy dài (nhiều σ × episode × MCTS search) có thể vài chục phút — giảm `--episodes`/`--sims`
  nếu muốn nhanh; `--no_cuda` nếu không có GPU (chậm hơn nhiều).